In [3]:
# install first
#pip install spacy pandas tqdm
#python -m spacy download fr_core_news_md

import pandas as pd
import spacy
from tqdm import tqdm

# ======================
# FICHIERS
# ======================
input_file = "La-ballade-de-Pern-intégrale-sentences.csv"
output_file = "La-ballade-de-Pern-intégrale-sentences-lemma.csv"

# ======================
# OPTIONS
# ======================
TEXT_COLUMN = "text"
LEMMA_COLUMN = "text_lemma"
BATCH_SIZE = 256

# Si True : les noms propres gardent leur forme originale
KEEP_PROPER_NOUNS = True

# Si True : on essaie de préserver l'aspect de casse
# ex. "Dragons" -> "Dragons" au lieu de "dragons"
PRESERVE_CASE = True

# ======================
# CHARGER SPACY
# ======================
# Installer une fois si besoin :
# pip install pandas spacy tqdm
# python -m spacy download fr_core_news_md

nlp = spacy.load("fr_core_news_md")
nlp.max_length = 5_000_000

# Désactiver ce qui n'est pas utile pour accélérer
# lemmatizer + tokenizer + pos restent nécessaires
disable_pipes = []
for pipe in ["ner", "parser", "textcat", "textcat_multilabel"]:
    if pipe in nlp.pipe_names:
        disable_pipes.append(pipe)

# ======================
# LIRE LE CSV
# ======================
df = pd.read_csv(input_file)

if TEXT_COLUMN not in df.columns:
    raise ValueError(f"La colonne '{TEXT_COLUMN}' est introuvable.")

# ======================
# FONCTIONS
# ======================
def match_case(original, lemma):
    """
    Adapte la casse du lemme à celle du token d'origine.
    """
    if not lemma:
        return lemma

    if original.isupper():
        return lemma.upper()

    # Titre / initiale majuscule
    if len(original) > 0 and original[0].isupper():
        return lemma[:1].upper() + lemma[1:]

    return lemma

def token_to_output(token):
    """
    Transforme un token en sortie lemmatisée.
    - conserve la ponctuation attachée via token.whitespace_
    - conserve les noms propres si demandé
    - préserve la casse si demandé
    """
    if token.is_space:
        return token.text

    # ponctuation : on la garde telle quelle
    if token.is_punct:
        return token.text + token.whitespace_

    # pronoms clitiques / traits d’union / apostrophes sont gérés par spaCy
    out = token.lemma_

    # parfois lemma_ est vide ou bizarre
    if not out or out == "":
        out = token.text

    # garder les noms propres tels quels
    if KEEP_PROPER_NOUNS and token.pos_ == "PROPN":
        out = token.text

    # préserver la casse
    elif PRESERVE_CASE:
        out = match_case(token.text, out)

    return out + token.whitespace_

def lemmatize_texts(texts):
    """
    Lemmatisation rapide par lots avec nlp.pipe.
    """
    results = []

    with nlp.select_pipes(disable=disable_pipes):
        for doc in tqdm(nlp.pipe(texts, batch_size=BATCH_SIZE), total=len(texts)):
            pieces = [token_to_output(token) for token in doc]
            lemma_text = "".join(pieces).strip()
            results.append(lemma_text)

    return results

# ======================
# PREPARER LES TEXTES
# ======================
texts = df[TEXT_COLUMN].fillna("").astype(str).tolist()

# ======================
# LEMMATISATION
# ======================
df[LEMMA_COLUMN] = lemmatize_texts(texts)

# ======================
# EXPORT
# ======================
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"Fichier créé : {output_file}")
print(f"Colonne ajoutée : {LEMMA_COLUMN}")

100%|█████████████████████████████████| 132850/132850 [01:21<00:00, 1621.52it/s]


Fichier créé : La-ballade-de-Pern-intégrale-sentences-lemma.csv
Colonne ajoutée : text_lemma
